# Summarising Datasets

In the explainer we just read, we saw how a single 480-minute wait dragged the mean up to 75 minutes while the median sat calmly at 32.5. The point was clear: the summary statistic we choose shapes the story the data tells.

Now we get to work with that idea in code. We have real-ish data from a hospital emergency department, and the hospital board wants a monthly performance summary. Our job is to decide which numbers belong in that summary and which would mislead.

By the end of this notebook we will be able to:

- Calculate **mean**, **median**, **mode**, **standard deviation**, **variance**, and **quantiles** for numeric columns
- Explain when the mean and median diverge and which to prefer for skewed data
- Use `.describe()`, `.value_counts()`, and `.groupby()` to summarise datasets
- Build a summary table suitable for a non-technical audience

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

---

## 1. Loading the data

The ED keeps its data across multiple files. Start with the two we will use most: patient arrivals and wait times.

In [ ]:
arrivals = pd.read_csv("../data/ed_arrivals.csv", parse_dates=["arrival_time"])
waits = pd.read_csv("../data/ed_wait_times.csv")

print(arrivals.shape)
arrivals.head()

In [ ]:
print(waits.shape)
waits.head()

Merge the two on `patient_id` so we can analyse arrivals and wait times together. This gives us one row per patient with both their demographics and their timings.

In [ ]:
df = arrivals.merge(waits, on="patient_id")
print(df.shape)
df.head()

---

## 2. A quick overview with `.describe()`

Before calculating anything specific, get the lay of the land. The `.describe()` method returns count, mean, standard deviation, min, quartiles, and max for every numeric column in one call. Think of it as the first handful of summary statistics from the explainer, computed automatically.

In [ ]:
df.describe()

Look at the count row. It is less than 5,000 for the wait columns, which means there are **missing values**. The other summary statistics are calculated on non-null rows only. Always check this before trusting any numbers `.describe()` gives us.

In [ ]:
df.isnull().sum()

---

## 3. Central tendency: mean, median, mode

We already know from the explainer that "average" is a vague word. There are three different measures of centre, and they can give very different answers depending on the shape of the data.

- `.mean()` returns the arithmetic average.
- `.median()` returns the middle value when sorted.
- `.mode()` returns the most frequent value.

For symmetric data, mean and median are close. For **skewed** data they diverge. Wait times in an ED are almost always right-skewed: most patients are seen quickly, but a few wait a very long time. Those long waits pull the mean upward, just like that 480-minute outlier in the explainer.

In [ ]:
print("Wait to treatment (minutes)")
print(f"  Mean:   {df['wait_to_treatment_min'].mean():.1f}")
print(f"  Median: {df['wait_to_treatment_min'].median():.1f}")
print(f"  Mode:   {df['wait_to_treatment_min'].mode().iloc[0]}")

The mean is higher than the median. That gap is the signature of a **right-skewed** distribution. When we report a "typical" wait time to the board, the median is the better choice here because it is not inflated by the long tail. Remember the ONS using the median for UK earnings for exactly the same reason.

In [ ]:
print("Total time in ED (minutes)")
print(f"  Mean:   {df['total_time_in_ed_min'].mean():.1f}")
print(f"  Median: {df['total_time_in_ed_min'].median():.1f}")

---

## 4. Spread: standard deviation, variance, and IQR

Knowing the centre is only half the picture. The explainer showed two departments with the same mean wait of 30 minutes but completely different patient experiences. Spread is what separates a reliable expectation from a misleading one.

- `.std()` returns the **standard deviation** — the typical distance of each value from the mean.
- `.var()` returns the **variance**, which is the standard deviation squared.
- The **interquartile range** (IQR) is Q3 minus Q1 and measures the spread of the middle 50% of the data.

In [ ]:
col = "wait_to_treatment_min"

print(f"Std dev:  {df[col].std():.1f}")
print(f"Variance: {df[col].var():.1f}")

### Quantiles and IQR

Use `.quantile()` to get specific percentiles. Q1 is the 25th percentile, Q3 is the 75th. The gap between them tells us where the middle half of patients fall.

In [ ]:
q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

print(f"Q1 (25th percentile): {q1:.1f}")
print(f"Q3 (75th percentile): {q3:.1f}")
print(f"IQR:                  {iqr:.1f}")

The IQR is also useful for identifying **outliers**. A common rule of thumb: any value below Q1 - 1.5 * IQR or above Q3 + 1.5 * IQR is considered an outlier. In an ED, these are patients who waited far longer than the norm.

In [ ]:
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = df[(df[col] < lower_fence) | (df[col] > upper_fence)]
print(f"Outlier fences: [{lower_fence:.1f}, {upper_fence:.1f}]")
print(f"Number of outliers: {len(outliers)}")

---

## 5. Counting categories with `.value_counts()`

Not everything in this dataset is numeric. Triage category, outcome, and arrival mode are all categorical. For these columns, `.value_counts()` counts how often each value appears. Add `normalize=True` to get proportions instead of raw counts.

In [ ]:
df["triage_category"].value_counts().sort_index()

In [ ]:
df["triage_category"].value_counts(normalize=True).sort_index().round(3)

In [ ]:
df["outcome"].value_counts()

In [ ]:
df["arrival_mode"].value_counts(normalize=True).round(3)

---

## 6. Summarising by group with `.groupby()`

A single median wait time for the whole department hides important variation. The board will want to know how wait times differ by triage category, because a 45-minute wait for a category 5 (minor) patient is very different from a 45-minute wait for a category 1 (resuscitation) patient.

`.groupby()` splits the data into groups, then we apply a summary function to each group.

In [ ]:
df.groupby("triage_category")["wait_to_treatment_min"].describe()

Triage categories 1 and 2 (resuscitation and emergency) should have much shorter waits. If they do not, that is a finding worth flagging to the board immediately.

In [ ]:
df.groupby("triage_category")["wait_to_treatment_min"].median()

In [ ]:
df.groupby("arrival_mode")["total_time_in_ed_min"].describe()

We can also group by multiple columns to build a cross-tabulation. This is useful when we want to see how two categorical variables interact.

In [ ]:
df.groupby(["triage_category", "outcome"]).size().unstack(fill_value=0)

---

## 7. Building a summary table for the board

The board does not want to read `.describe()` output. They want a clean table with metrics that matter, presented in language they can act on. This is the "choice of statistic changes the story" principle from the explainer, put into practice.

Here is how to build a monthly summary by extracting the month from the arrival time and aggregating.

In [ ]:
df["month"] = df["arrival_time"].dt.month

monthly = df.groupby("month").agg(
    total_patients=("patient_id", "count"),
    median_wait_to_triage=("wait_to_triage_min", "median"),
    median_wait_to_treatment=("wait_to_treatment_min", "median"),
    median_total_time=("total_time_in_ed_min", "median"),
    pct_admitted=("outcome", lambda x: (x == "admitted").mean() * 100),
    pct_left_without_treatment=("outcome", lambda x: (x == "left_without_treatment").mean() * 100),
).round(1)

monthly

This table shows monthly patient volumes, median wait times, and the percentage of patients admitted or leaving without treatment. Notice the deliberate use of medians rather than means. A board member can scan this in seconds and spot months where performance dipped.

---

## 8. Exercises

Time to practise. Complete each exercise in the code cell provided.

### Exercise 1: Compare mean and median

Calculate the mean and median for `wait_to_triage_min`. Print both values and write a comment stating which one you would report to the board and why.

In [ ]:
# Your code here


### Exercise 2: IQR and outliers for total ED time

Calculate Q1, Q3, and the IQR for `total_time_in_ed_min`. Use the 1.5 * IQR rule to count how many patients are outliers on the high end (above the upper fence). Print the upper fence value and the outlier count.

In [ ]:
# Your code here


### Exercise 3: Grouped summary by age group

Use `.groupby()` to calculate the median `wait_to_treatment_min` and median `total_time_in_ed_min` for each `age_group`. Which age group has the longest median total time?

In [ ]:
# Your code here


### Exercise 4: Board-ready summary by triage category

Create a summary table grouped by `triage_category` that shows:

- Number of patients
- Median wait to treatment
- Median total time in ED
- Percentage discharged
- Percentage admitted

Round all values to one decimal place.

In [ ]:
# Your code here


---

## Summary

In this notebook we:

- Used `.describe()` to get a quick overview of numeric columns
- Calculated **mean**, **median**, and **mode** and saw how skewness makes them diverge
- Measured spread with **standard deviation**, **variance**, and **IQR**
- Used the 1.5 * IQR rule to identify **outliers**
- Counted categorical values with `.value_counts()` and `normalize=True`
- Split data into groups with `.groupby()` and applied summary functions
- Built a clean monthly summary table, choosing medians over means for a skewed dataset

These are the building blocks. But summary statistics compress data, and compression always hides something. In the next notebook we will look at the **shape** of these distributions using histograms and box plots, so we can see what the numbers alone cannot tell us.